# Metadata archiver

This notebook creates an archive of the YouTube metadata retrieved with PYG lean via the API.
The results are stored in a folder specified by the user below.

In [ ]:
import os
import zipfile
import shutil
import json
from datetime import datetime
from collections import defaultdict

# Flags

In [ ]:
# Specify actual directory paths
zip_directory = 'D:\\JPMETA'
extract_directory = 'D:\\JPMETA\\extracted'
output_directory = 'D:\\JPMETA\\meta'

In [2]:


# Step 1: Get a sorted list of zip files by their modification time
def get_sorted_zip_files(directory, log_file):
    zip_files = []
    for file in os.listdir(directory):
        if file.endswith('.zip'):
            file_path = os.path.join(directory, file)
            timestamp = os.path.getmtime(file_path)  # Get the modification time
            zip_files.append((file_path, timestamp))
    # Sort by modification time
    zip_files.sort(key=lambda x: x[1])

    # Log the sorted zip files
    with open(log_file, 'a', encoding='utf-8') as log:
        log.write("=== Sorted ZIP Files ===\n")
        for file_path, timestamp in zip_files:
            log.write(f"{file_path}, timestamp: {datetime.fromtimestamp(timestamp).isoformat()}\n")
        log.write("\n")
    return zip_files

# Step 2: Extract metadata from a single zip file
def extract_metadata_from_zip(zip_file, log_file):
    metadata = []
    extraction_dir = 'extracted'  # Temporary directory for extraction
    if os.path.exists(extraction_dir):
        # Clean up the extraction directory before processing a new zip file
        for file in os.listdir(extraction_dir):
            file_path = os.path.join(extraction_dir, file)
            if os.path.isfile(file_path):  # If it's a file, delete it
                os.remove(file_path)
            elif os.path.isdir(file_path):  # If it's a directory, delete it
                shutil.rmtree(file_path)

    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(extraction_dir)
        metadata_folder = os.path.join(extraction_dir, 'video_meta')
        if not os.path.exists(metadata_folder):
            with open(log_file, 'a', encoding='utf-8') as log:
                log.write(f"Metadata folder not found in {zip_file}. Skipping...\n")
            return metadata

        extracted_files = os.listdir(metadata_folder)

        # Log the extracted files
        with open(log_file, 'a', encoding='utf-8') as log:
            log.write(f"=== Extracted from {zip_file} ===\n")
            log.write(f"Extracted files: {extracted_files}\n")

        for json_file in extracted_files:
            if json_file.endswith('.json'):
                json_path = os.path.join(metadata_folder, json_file)
                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    video_id = json_file.split('.')[0]
                    if "items" in data and len(data["items"]) > 0:  # Ensure videoId exists
                        metadata.append((video_id, data))
                        # Log successful extraction
                        with open(log_file, 'a', encoding='utf-8') as log:
                            log.write(f"Processed {json_file} for videoId: {video_id} (from {zip_file})\n")
                    else:
                        # Log missing or invalid data
                        with open(log_file, 'a', encoding='utf-8') as log:
                            log.write(f"Skipped {json_file}: No valid items found (from {zip_file})\n")
    return metadata

# Step 3: Merge metadata for each videoId and save as JSON files
def merge_metadata_from_zips(sorted_zip_files, log_file, output_directory):
    merged_metadata = defaultdict(lambda: {"meta_data": {}, "tracked_changes": defaultdict(list)})
    tracked_keys = ["viewCount", "likeCount", "commentCount"]  # Keys to track changes

    for zip_file, timestamp in sorted_zip_files:
        formatted_timestamp = datetime.fromtimestamp(timestamp).isoformat()
        metadata = extract_metadata_from_zip(zip_file, log_file)

        for video_id, data in metadata:
            # Initialize the entry if it doesn't exist
            if not merged_metadata[video_id]["meta_data"]:
                merged_metadata[video_id]["meta_data"] = {
                    "videoId": video_id,
                    "publishedAt": data["items"][0]["snippet"].get("publishedAt", "missing"),
                    "channelId": data["items"][0]["snippet"].get("channelId", "missing"),
                    "title": data["items"][0]["snippet"].get("title", "missing"),
                    "description": data["items"][0]["snippet"].get("description", "missing"),
                    "channelTitle": data["items"][0]["snippet"].get("channelTitle", "missing"),
                    "tags": data["items"][0]["snippet"].get("tags", "missing"),
                    "categoryId": data["items"][0]["snippet"].get("categoryId", "missing"),
                    "liveBroadcastContent": data["items"][0]["snippet"].get("liveBroadcastContent", "missing"),
                    "defaultLanguage": data["items"][0]["snippet"].get("defaultLanguage", "missing"),
                    "duration": data["items"][0]["contentDetails"].get("duration", "missing"),
                    "dimension": data["items"][0]["contentDetails"].get("dimension", "missing"),
                    "definition": data["items"][0]["contentDetails"].get("definition", "missing"),
                    "caption": data["items"][0]["contentDetails"].get("caption", "missing"),
                    "licensedContent": data["items"][0]["contentDetails"].get("licensedContent", "missing"),
                    "projection": data["items"][0]["contentDetails"].get("projection", "missing"),
                    "uploadStatus": data["items"][0]["status"].get("uploadStatus", "missing"),
                    "privacyStatus": data["items"][0]["status"].get("privacyStatus", "missing"),
                    "license": data["items"][0]["status"].get("license", "missing"),
                    "embeddable": data["items"][0]["status"].get("embeddable", "missing"),
                    "publicStatsViewable": data["items"][0]["status"].get("publicStatsViewable", "missing"),
                    "madeForKids": data["items"][0]["status"].get("madeForKids", "missing")
                }

            # Track changes for specific keys
            for key in tracked_keys:
                value = data["items"][0]["statistics"].get(key, "missing")
                if value != "missing":
                    # Ensure the timestamp and value are unique
                    existing_timestamps = [entry["timestamp"] for entry in merged_metadata[video_id]["tracked_changes"][key]]
                    if formatted_timestamp not in existing_timestamps:
                        merged_metadata[video_id]["tracked_changes"][key].append(
                            {"value": value, "timestamp": formatted_timestamp}
                        )
                        # Log tracked changes
                        with open(log_file, 'a', encoding='utf-8') as log:
                            log.write(f"Tracked change for {video_id}: {key} = {value} at {formatted_timestamp} (from {zip_file})\n")

    # Finalize the metadata
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)

    for video_id, entry in merged_metadata.items():
        meta_data = entry["meta_data"]
        for key in tracked_keys:
            if entry["tracked_changes"][key]:
                # Sort changes by timestamp
                entry["tracked_changes"][key].sort(key=lambda x: x["timestamp"])
                meta_data[key] = entry["tracked_changes"][key]

        # Save each videoId's metadata to a JSON file
        output_file = os.path.join(output_directory, f"{video_id}.json")
        with open(output_file, 'w', encoding='utf-8') as out_file:
            json.dump(meta_data, out_file, ensure_ascii=False, indent=4)

        # Log saved file
        with open(log_file, 'a', encoding='utf-8') as log:
            log.write(f"Saved metadata for videoId {video_id} to {output_file}\n")

In [3]:
# Main program
def main(directory, output_directory, log_file):
    with open(log_file, 'w', encoding='utf-8') as log:
        log.write("=== Metadata Processing Log ===\n")
    sorted_zip_files = get_sorted_zip_files(directory, log_file)
    merge_metadata_from_zips(sorted_zip_files, log_file, output_directory)
    print("Metadata processing complete. Check logs for details.")

# Execution Section

In [ ]:
# Run the program
zip_directory = 'D:\\JPMETA'
extract_directory = 'D:\\JPMETA\\extracted'
output_directory = 'D:\\JPMETA\\meta'

main(zip_directory, output_directory, f'{zip_directory}\\processing_log.txt')